# AI-CDFI — Cureus Journal Reproducibility Pipeline

This notebook implements the reproducibility pipeline for the Cureus Journal of Computer Science manuscript on CSE-CIC-IDS2018:

1. Leakage-controlled source-only Transformer baseline
2. Same-day evaluation
3. Full 6×6 cross-day generalisation matrix
4. Unsupervised DANN domain adaptation using **unlabelled target adaptation data**
5. Purged temporal target adaptation/test split
6. Source-validation threshold recalibration after adaptation
7. DANN sensitivity analysis over domain-loss strength
8. Simple supervised target fine-tuning as an **upper-bound comparator** (not claimed as unsupervised DA)
9. ROC/PR curves and confusion matrices
10. Integrated Gradients XAI without Captum
11. Feature-level and temporal XAI plots
12. Temporal forensic graph reconstruction
13. Training curves
14. Automatic CSV/PNG/JSON/GraphML outputs
15. Final comparison tables and manuscript-ready summary files

**Important:** the target-test labels are never used to train or adapt the unsupervised DANN model. They are used only for final evaluation.


In [ ]:

# ================================================================
# 1. CONFIGURATION
# ================================================================
import os, re, glob, json, math, random, gc, warnings, time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    precision_recall_fscore_support, roc_auc_score,
    average_precision_score, confusion_matrix, accuracy_score,
    roc_curve, precision_recall_curve
)

import joblib
import networkx as nx

warnings.filterwarnings("ignore")

SEED = 42
DATASET_ID = "solarmainframe/ids-intrusion-csv"
OUT = Path("/kaggle/working/ai_cdfi_cureus_reproducibility")
OUT.mkdir(parents=True, exist_ok=True)

DAYS = ["02-14-2018","02-15-2018","02-16-2018",
        "02-22-2018","02-28-2018","03-01-2018"]

# Main DA pairs: selected to cover different drift types while controlling runtime.
ADAPTATION_PAIRS = [
    ("02-14-2018","02-15-2018"),
    ("02-14-2018","02-22-2018"),
    ("02-14-2018","02-28-2018"),
    ("02-15-2018","02-16-2018"),
    ("02-28-2018","03-01-2018"),
]

# Reproducibility controls. Increase after pipeline validation for the final manuscript run.
MAX_ROWS_PER_DAY = None
MAX_TRAIN_WINDOWS = 60000
MAX_VAL_WINDOWS = 15000
MAX_TEST_WINDOWS = 40000
MAX_ADAPT_WINDOWS = 30000

SEQ_LEN = 32
SEQ_STRIDE = 8
BLOCK_SIZE = 200
TRAIN_RATIO, VAL_RATIO, TEST_RATIO = 0.70, 0.15, 0.15

DANN_ADAPT_RATIO = 0.50
PURGE_WINDOWS = SEQ_LEN - 1

BATCH_SIZE = 128
EPOCHS = 20
ADAPT_EPOCHS = 15
FINE_TUNE_EPOCHS = 5

LR = 1e-4
DANN_LR = 1e-4
FINE_TUNE_LR = 5e-5
PATIENCE = 5

D_MODEL, N_HEADS, N_LAYERS, FF_DIM, DROPOUT = 128, 4, 3, 256, 0.15

# Primary DANN setting
LAMBDA_DOMAIN = 0.10

# Sensitivity experiment: intentionally small and efficient.
DANN_LAMBDA_SWEEP = [0.05, 0.10, 0.20]

# XAI
XAI_SAMPLES = 12
IG_STEPS = 16

# Graph
WINDOW_SECONDS = 300

NUM_WORKERS = 2
PIN_MEMORY = True

VARIANCE_THRESHOLD = 1e-6
CORRELATION_THRESHOLD = 0.95
LEAKAGE_AUC_THRESHOLD = 0.985

RUN_BASELINE = True
RUN_CROSS_DAY = True
RUN_DANN = True
RUN_DANN_SWEEP = True
RUN_SUPERVISED_TARGET_FT = True
RUN_XAI = True
RUN_GRAPHS = True
RUN_PLOTS = True

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("AI-CDFI Cureus reproducibility pipeline")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("Output:", OUT)


In [ ]:

# ================================================================
# 2. CAPTUM-FREE INTEGRATED GRADIENTS
# ================================================================
class IntegratedGradientsTorch:
    def __init__(self, model):
        self.model = model

    def attribute(self, inputs, baselines=None, n_steps=16):
        if baselines is None:
            baselines = torch.zeros_like(inputs)

        was_training = self.model.training
        self.model.eval()

        inputs = inputs.detach()
        baselines = baselines.detach()
        delta = inputs - baselines
        total_grad = torch.zeros_like(inputs)

        for k in range(n_steps):
            alpha = (k + 0.5) / n_steps
            x = (baselines + alpha * delta).detach().requires_grad_(True)

            self.model.zero_grad(set_to_none=True)
            out = self.model(x)

            if out.ndim == 0:
                objective = out
            elif out.ndim == 1:
                objective = out.sum()
            elif out.ndim == 2:
                objective = out[:, 0].sum()
            else:
                raise ValueError("Explained model must return scalar, [B], or [B,1].")

            grad = torch.autograd.grad(
                objective, x, retain_graph=False,
                create_graph=False, allow_unused=False
            )[0]

            total_grad += grad.detach()
            del x, out, grad

        attr = delta * (total_grad / n_steps)

        if was_training:
            self.model.train()

        return attr.detach()

print("Integrated Gradients loaded — no Captum installation required.")


In [ ]:

# ================================================================
# 3. DATA DISCOVERY AND CLEANING
# ================================================================
def discover_csv_files():
    files = []
    for root in ["/kaggle/input", "/kaggle/working"]:
        if os.path.exists(root):
            for ext in ("*.csv", "*.CSV"):
                files.extend(glob.glob(os.path.join(root, "**", ext), recursive=True))
    return sorted(set(files))

CSV_FILES = discover_csv_files()

if not CSV_FILES:
    raise FileNotFoundError(
        "No CSV files found. In Kaggle use Add Input and attach the CSE-CIC-IDS2018 CSV dataset."
    )

print("CSV files found:", len(CSV_FILES))
for p in CSV_FILES[:10]:
    print(" ", p)

LABEL_CANDIDATES = ["Label","label","Attack","attack","Class","class","Category","category"]
TIMESTAMP_CANDIDATES = ["Timestamp","timestamp","Time","time","Date","date"]

def find_column(columns, candidates):
    norm = {str(c).strip().lower(): c for c in columns}
    for x in candidates:
        if x.lower() in norm:
            return norm[x.lower()]
    for c in columns:
        cl = str(c).strip().lower()
        if any(x.lower() in cl for x in candidates):
            return c
    return None

def find_csv(day):
    token = day.lower()
    hits = [p for p in CSV_FILES if token in os.path.basename(p).lower()]
    if not hits:
        m,d,y = day.split("-")
        alt = f"{y}-{m}-{d}".lower()
        hits = [p for p in CSV_FILES if alt in os.path.basename(p).lower()]
    if not hits:
        raise FileNotFoundError(f"No CSV found for {day}.")
    # Prefer exact day-level mirror names.
    hits.sort(key=lambda p: (
        0 if os.path.basename(p).lower() == f"{day}.csv" else 1,
        len(p)
    ))
    return hits[0]

def clean_day(day, max_rows=None):
    path = find_csv(day)
    print(f"\nLoading {day}: {path}")

    df = pd.read_csv(path, low_memory=False)
    df.columns = [str(c).strip() for c in df.columns]

    label_col = find_column(df.columns, LABEL_CANDIDATES)
    ts_col = find_column(df.columns, TIMESTAMP_CANDIDATES)

    if label_col is None:
        raise ValueError(f"{day}: label column not found.")

    excluded = {label_col}
    if ts_col:
        excluded.add(ts_col)

    candidates = [c for c in df.columns if c not in excluded]

    numeric = df[candidates].apply(pd.to_numeric, errors="coerce")
    valid = ~numeric.isna().all(axis=1)

    df = df.loc[valid].copy()
    numeric = numeric.loc[valid]
    df[candidates] = numeric

    df = df.dropna(subset=[label_col]).copy()

    labels = df[label_col].astype(str).str.strip()
    df["target"] = (~labels.str.upper().eq("BENIGN")).astype(np.int8)

    if ts_col:
        ts = pd.to_datetime(df[ts_col], errors="coerce")
        if ts.isna().mean() > 0.50:
            ts = pd.to_datetime(df[ts_col], errors="coerce", dayfirst=True)
        df[ts_col] = ts
        df = df.dropna(subset=[ts_col]).sort_values(ts_col).reset_index(drop=True)
    else:
        df = df.reset_index(drop=True)

    df = df.drop_duplicates().reset_index(drop=True)

    df["event_id"] = [f"{day}-EVD-{i:08d}" for i in range(len(df))]

    if max_rows is not None and len(df) > max_rows:
        idx = np.linspace(0, len(df)-1, max_rows, dtype=int)
        df = df.iloc[idx].reset_index(drop=True)

    print(
        f"rows={len(df):,} | anomalies={int(df.target.sum()):,} | "
        f"anomaly_rate={df.target.mean():.4%}"
    )
    return df, label_col, ts_col

print("Data loader ready.")


In [ ]:

# ================================================================
# 4. LEAKAGE-CONTROLLED FEATURE SELECTION + PREPROCESSING
# ================================================================
def block_split_indices(n, block_size=BLOCK_SIZE):
    starts = np.arange(0, n, block_size)
    assignment = np.arange(len(starts)) % 10

    tr_mask = assignment < 7
    va_mask = (assignment >= 7) & (assignment < 8.5)
    # Integer block assignment: 7 blocks train, 1 validation, 2 test.
    # This is deterministic and preserves temporal locality.
    tr_mask = assignment <= 6
    va_mask = assignment == 7
    te_mask = assignment >= 8

    def expand(mask):
        idx = []
        for s, use in zip(starts, mask):
            if use:
                idx.extend(range(s, min(s + block_size, n)))
        return np.asarray(idx, dtype=np.int64)

    return expand(tr_mask), expand(va_mask), expand(te_mask)

def select_features_source(df, label_col):
    excluded = {"target", "event_id", label_col}
    ts_col = find_column(df.columns, TIMESTAMP_CANDIDATES)
    if ts_col:
        excluded.add(ts_col)

    candidates = [c for c in df.columns if c not in excluded]

    # Remove obvious identifier/port fields from the model input.
    hard = re.compile(
        r"(^flow.?id$|src.?port|dst.?port|source.?port|destination.?port|protocol$)",
        re.I
    )
    candidates = [c for c in candidates if not hard.search(str(c))]

    X = df[candidates].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan)
    X = X.dropna(axis=1, how="all")

    med = X.median(numeric_only=True)
    X = X.fillna(med).fillna(0)

    var = X.var()
    keep = var[var > VARIANCE_THRESHOLD].index.tolist()
    X = X[keep]

    if len(X.columns) > 1:
        corr = X.corr().abs()
        upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        drop = [c for c in upper.columns if any(upper[c] >= CORRELATION_THRESHOLD)]
        X = X[[c for c in X.columns if c not in drop]]

    leakage = []
    y = df["target"].values

    for c in X.columns:
        try:
            auc = roc_auc_score(y, X[c].values)
            auc = max(auc, 1 - auc)
            if auc >= LEAKAGE_AUC_THRESHOLD:
                leakage.append(c)
        except Exception:
            pass

    keep = [c for c in X.columns if c not in leakage]

    print(
        f"Feature selection: {len(candidates)} initial -> "
        f"{len(keep)} retained; leakage removed={len(leakage)}"
    )
    return keep

def preprocess_matrix(df, feature_cols, scaler=None, fit_scaler=False,
                      fit_rows=None, medians=None):

    X = df[feature_cols].copy()

    for c in feature_cols:
        X[c] = pd.to_numeric(X[c], errors="coerce")

    X = X.replace([np.inf, -np.inf], np.nan)

    if medians is None:
        fit_df = X if fit_rows is None else X.iloc[np.asarray(fit_rows)]
        medians = fit_df.median(numeric_only=True).to_dict()

    X = X.fillna(pd.Series(medians)).fillna(0)

    Xv = X.values.astype(np.float32)

    if scaler is None:
        scaler = StandardScaler()

    if fit_scaler:
        fit_idx = np.arange(len(Xv)) if fit_rows is None else np.asarray(fit_rows)
        scaler.fit(Xv[fit_idx])

    Xv = scaler.transform(Xv).astype(np.float32)

    return Xv, scaler, medians

class WindowDataset(Dataset):
    def __init__(self, X, y, positions, seq_len=SEQ_LEN):
        self.X = X
        self.y = y
        self.positions = np.asarray(positions, dtype=np.int64)
        self.seq_len = seq_len

    def __len__(self):
        return len(self.positions)

    def __getitem__(self, i):
        end = int(self.positions[i])
        start = end - self.seq_len + 1
        return (
            torch.from_numpy(self.X[start:end+1]).float(),
            torch.tensor(int(self.y[end]), dtype=torch.long)
        )

def valid_window_positions(n, candidate_indices):
    candidate = np.asarray(candidate_indices, dtype=np.int64)
    if len(candidate) == 0:
        return candidate

    mask = np.zeros(n, dtype=bool)
    mask[candidate] = True

    out = []
    for end in np.flatnonzero(mask):
        if end < SEQ_LEN - 1:
            continue
        if mask[end-SEQ_LEN+1:end+1].all():
            out.append(end)

    return np.asarray(out, dtype=np.int64)

def cap_positions(pos, max_n):
    if max_n is None or len(pos) <= max_n:
        return pos
    # Deterministic evenly spaced sampling.
    return pos[np.linspace(0, len(pos)-1, max_n, dtype=int)]

print("Preprocessing functions ready.")


In [ ]:

# ================================================================
# 5. TRANSFORMER + DANN
# ================================================================
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=4096):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).float().unsqueeze(1)
        div = torch.exp(
            torch.arange(0, d_model, 2).float()
            * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0))

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class ForensicTransformer(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.input_proj = nn.Linear(input_dim, D_MODEL)
        self.pos = PositionalEncoding(D_MODEL)

        layer = nn.TransformerEncoderLayer(
            d_model=D_MODEL,
            nhead=N_HEADS,
            dim_feedforward=FF_DIM,
            dropout=DROPOUT,
            batch_first=True,
            norm_first=True
        )
        self.encoder = nn.TransformerEncoder(layer, N_LAYERS)
        self.norm = nn.LayerNorm(D_MODEL)
        self.classifier = nn.Linear(D_MODEL, 2)

    def representation(self, x):
        z = self.input_proj(x)
        z = self.pos(z)
        z = self.encoder(z)
        return self.norm(z[:, -1, :])

    def forward(self, x):
        return self.classifier(self.representation(x))

class GradReverse(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambd):
        ctx.lambd = lambd
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad):
        return -ctx.lambd * grad, None

def grad_reverse(x, lambd):
    return GradReverse.apply(x, lambd)

class DANNTransformer(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.backbone = ForensicTransformer(input_dim)
        self.domain = nn.Sequential(
            nn.Linear(D_MODEL, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 2)
        )

    def forward(self, x, lambd=0.0):
        rep = self.backbone.representation(x)
        cls = self.backbone.classifier(rep)
        dom = self.domain(grad_reverse(rep, lambd))
        return cls, dom

print("Transformer and DANN models ready.")


In [ ]:

# ================================================================
# 6. METRICS, LOADERS, TRAINING
# ================================================================
def class_weights(y):
    counts = np.bincount(y, minlength=2).astype(np.float32)
    w = counts.sum() / (2 * np.maximum(counts, 1))
    return torch.tensor(w, dtype=torch.float32, device=DEVICE)

def make_loader(d, positions, shuffle=False):
    return DataLoader(
        WindowDataset(d["X"], d["y"], positions),
        batch_size=BATCH_SIZE,
        shuffle=shuffle,
        num_workers=NUM_WORKERS,
        pin_memory=PIN_MEMORY,
        persistent_workers=(NUM_WORKERS > 0)
    )

def predict(model, loader, dann=False):
    model.eval()
    probs, ys = [], []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            logits = model(xb)[0] if dann else model(xb)
            probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
            ys.extend(yb.numpy())

    return np.asarray(ys), np.asarray(probs)

def best_threshold(y, p):
    best_t, best_f1 = 0.5, -1
    for t in np.linspace(0.01, 0.99, 99):
        pred = (p >= t).astype(int)
        _, _, f1, _ = precision_recall_fscore_support(
            y, pred, average="binary", zero_division=0
        )
        if f1 > best_f1:
            best_t, best_f1 = float(t), float(f1)
    return best_t, best_f1

def metric_dict(y, p, threshold=0.5):
    pred = (p >= threshold).astype(int)
    pr, re, f1, _ = precision_recall_fscore_support(
        y, pred, average="binary", zero_division=0
    )
    cm = confusion_matrix(y, pred, labels=[0, 1])

    return {
        "precision": float(pr),
        "recall": float(re),
        "f1": float(f1),
        "accuracy": float(accuracy_score(y, pred)),
        "roc_auc": float(roc_auc_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "pr_auc": float(average_precision_score(y, p)) if len(np.unique(y)) == 2 else np.nan,
        "tn": int(cm[0,0]), "fp": int(cm[0,1]),
        "fn": int(cm[1,0]), "tp": int(cm[1,1]),
        "threshold": float(threshold)
    }

def train_baseline(model, train_loader, val_loader, y_train, out_path):
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss(weight=class_weights(y_train))

    best = float("inf")
    wait = 0
    history = []

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total, n = 0.0, 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += loss.item() * len(yb)
            n += len(yb)

        model.eval()
        val_total, vn = 0.0, 0

        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(DEVICE, non_blocking=True)
                yb = yb.to(DEVICE, non_blocking=True)
                loss = crit(model(xb), yb)
                val_total += loss.item() * len(yb)
                vn += len(yb)

        tr_loss = total / max(n, 1)
        va_loss = val_total / max(vn, 1)

        history.append({
            "epoch": epoch,
            "train_loss": tr_loss,
            "val_loss": va_loss
        })

        print(f"Epoch {epoch:02d}/{EPOCHS} | train={tr_loss:.4f} | val={va_loss:.4f}")

        if va_loss < best - 1e-4:
            best = va_loss
            wait = 0
            torch.save(
                {"model_state_dict": model.state_dict(), "history": history},
                out_path
            )
        else:
            wait += 1
            if wait >= PATIENCE:
                print("Early stopping.")
                break

    model.load_state_dict(
        torch.load(out_path, map_location=DEVICE)["model_state_dict"]
    )
    return history

def train_dann(model, src_loader, tgt_loader, y_src, out_path, lambda_max):
    opt = torch.optim.AdamW(model.parameters(), lr=DANN_LR, weight_decay=1e-4)
    cls_loss = nn.CrossEntropyLoss(weight=class_weights(y_src))
    dom_loss = nn.CrossEntropyLoss()

    history = []

    for epoch in range(1, ADAPT_EPOCHS + 1):
        model.train()
        total, n = 0.0, 0
        target_iter = iter(tgt_loader)

        progress = epoch / ADAPT_EPOCHS
        lambd = lambda_max * (2 / (1 + math.exp(-10 * progress)) - 1)

        for xs, ys in src_loader:
            try:
                xt, _ = next(target_iter)
            except StopIteration:
                target_iter = iter(tgt_loader)
                xt, _ = next(target_iter)

            xs = xs.to(DEVICE, non_blocking=True)
            ys = ys.to(DEVICE, non_blocking=True)
            xt = xt.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            src_cls, src_dom = model(xs, lambd)
            _, tgt_dom = model(xt, lambd)

            y_dom_src = torch.zeros(len(xs), dtype=torch.long, device=DEVICE)
            y_dom_tgt = torch.ones(len(xt), dtype=torch.long, device=DEVICE)

            loss_cls = cls_loss(src_cls, ys)
            loss_dom = 0.5 * (
                dom_loss(src_dom, y_dom_src) +
                dom_loss(tgt_dom, y_dom_tgt)
            )

            loss = loss_cls + loss_dom
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total += loss.item() * len(ys)
            n += len(ys)

        epoch_loss = total / max(n, 1)
        history.append({
            "epoch": epoch,
            "loss": epoch_loss,
            "lambda": lambd
        })

        print(
            f"DANN epoch {epoch:02d}/{ADAPT_EPOCHS} | "
            f"loss={epoch_loss:.4f} | lambda={lambd:.3f}"
        )

    torch.save(
        {"model_state_dict": model.state_dict(), "history": history},
        out_path
    )
    return history

print("Training and evaluation utilities ready.")


In [ ]:

# ================================================================
# 7. DAY PREPARATION
# ================================================================
def prepare_day(day, source_feature_cols=None, source_scaler=None,
                source_medians=None, fit_scaler=False):

    df, label_col, ts_col = clean_day(day, MAX_ROWS_PER_DAY)

    row_tr, row_va, row_te = block_split_indices(len(df))

    if source_feature_cols is None:
        feature_cols = select_features_source(
            df.iloc[row_tr].copy(), label_col
        )
    else:
        feature_cols = list(source_feature_cols)

    # Target must use exactly the source feature schema.
    for c in feature_cols:
        if c not in df.columns:
            df[c] = 0.0

    X, scaler, medians = preprocess_matrix(
        df,
        feature_cols,
        scaler=source_scaler,
        fit_scaler=fit_scaler,
        fit_rows=row_tr,
        medians=source_medians
    )

    y = df["target"].values.astype(np.int64)

    tr_pos = cap_positions(
        valid_window_positions(len(df), row_tr),
        MAX_TRAIN_WINDOWS
    )
    va_pos = cap_positions(
        valid_window_positions(len(df), row_va),
        MAX_VAL_WINDOWS
    )
    te_pos = cap_positions(
        valid_window_positions(len(df), row_te),
        MAX_TEST_WINDOWS
    )

    return {
        "day": day,
        "df": df,
        "features": feature_cols,
        "scaler": scaler,
        "medians": medians,
        "X": X,
        "y": y,
        "train_pos": tr_pos,
        "val_pos": va_pos,
        "test_pos": te_pos,
        "label_col": label_col,
        "ts_col": ts_col
    }

ARTIFACTS = {}
SOURCE_THRESHOLDS = {}
BASELINE_MODELS = {}
BASELINE_HISTORY = {}

print("Day preparation ready.")


In [ ]:

# ================================================================
# 8. SAME-DAY BASELINE
# ================================================================
baseline_rows = []

if RUN_BASELINE:
    print("\n" + "="*90)
    print("SAME-DAY TRANSFORMER BASELINE")
    print("="*90)

    for day in DAYS:
        print(f"\n--- BASELINE {day} ---")

        d = prepare_day(day, fit_scaler=True)
        ARTIFACTS[day] = d

        tr_loader = make_loader(d, d["train_pos"], shuffle=True)
        va_loader = make_loader(d, d["val_pos"])
        te_loader = make_loader(d, d["test_pos"])

        model = ForensicTransformer(len(d["features"])).to(DEVICE)
        ckpt = OUT / f"baseline_{day}.pt"

        history = train_baseline(
            model, tr_loader, va_loader,
            d["y"][d["train_pos"]],
            str(ckpt)
        )

        yv, pv = predict(model, va_loader)
        threshold, val_f1 = best_threshold(yv, pv)

        yt, pt = predict(model, te_loader)
        m = metric_dict(yt, pt, threshold)

        m.update({
            "day": day,
            "model": "Transformer-baseline",
            "n_features": len(d["features"]),
            "val_f1": val_f1
        })

        baseline_rows.append(m)
        SOURCE_THRESHOLDS[day] = threshold
        BASELINE_MODELS[day] = model
        BASELINE_HISTORY[day] = history

        pd.DataFrame(history).to_csv(
            OUT / f"baseline_history_{day}.csv", index=False
        )
        joblib.dump(d["scaler"], OUT / f"scaler_{day}.joblib")

        with open(OUT / f"features_{day}.json", "w") as f:
            json.dump(d["features"], f, indent=2)

        with open(OUT / f"baseline_threshold_{day}.json", "w") as f:
            json.dump({"threshold": threshold, "validation_f1": val_f1}, f, indent=2)

        print(
            f"TEST | F1={m['f1']:.4f} | "
            f"ROC-AUC={m['roc_auc']:.4f} | "
            f"PR-AUC={m['pr_auc']:.4f} | threshold={threshold:.2f}"
        )

        del tr_loader, va_loader, te_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

baseline_df = pd.DataFrame(baseline_rows)
baseline_df.to_csv(OUT / "baseline_same_day_results.csv", index=False)

print("\nBASELINE SUMMARY")
display(baseline_df[["day","f1","roc_auc","pr_auc","precision","recall"]])


In [ ]:

# ================================================================
# 9. FULL CROSS-DAY GENERALISATION MATRIX
# ================================================================
def evaluate_cross_day(train_day, test_day):
    src = ARTIFACTS[train_day]

    model = ForensicTransformer(len(src["features"])).to(DEVICE)
    model.load_state_dict(
        torch.load(
            OUT / f"baseline_{train_day}.pt",
            map_location=DEVICE
        )["model_state_dict"]
    )

    # Source feature schema, source medians and source scaler are reused.
    tgt = prepare_day(
        test_day,
        source_feature_cols=src["features"],
        source_scaler=src["scaler"],
        source_medians=src["medians"],
        fit_scaler=False
    )

    # Cross-day evaluation uses the target day only for inference.
    # No target labels/features are fitted into preprocessing.
    all_pos = cap_positions(
        valid_window_positions(
            len(tgt["df"]), np.arange(len(tgt["df"]))
        ),
        MAX_TEST_WINDOWS
    )

    loader = make_loader(tgt, all_pos)

    y, p = predict(model, loader)
    m = metric_dict(y, p, SOURCE_THRESHOLDS.get(train_day, 0.5))

    m.update({
        "train_day": train_day,
        "test_day": test_day
    })

    return m

cross_rows = []

if RUN_CROSS_DAY:
    print("\n" + "="*90)
    print("FULL CROSS-DAY GENERALISATION")
    print("="*90)

    for source in DAYS:
        for target in DAYS:
            if source == target:
                continue

            print(f"{source} -> {target}")

            try:
                cross_rows.append(
                    evaluate_cross_day(source, target)
                )
            except Exception as e:
                print("FAILED:", e)

cross_df = pd.DataFrame(cross_rows)
cross_df.to_csv(OUT / "cross_day_results_long.csv", index=False)

cross_f1 = cross_df.pivot(
    index="train_day", columns="test_day", values="f1"
)
cross_auc = cross_df.pivot(
    index="train_day", columns="test_day", values="roc_auc"
)
cross_pr = cross_df.pivot(
    index="train_day", columns="test_day", values="pr_auc"
)

cross_f1.to_csv(OUT / "cross_day_f1_matrix.csv")
cross_auc.to_csv(OUT / "cross_day_roc_auc_matrix.csv")
cross_pr.to_csv(OUT / "cross_day_pr_auc_matrix.csv")

print("\nCross-day F1 matrix")
display(cross_f1.round(4))

print("\nCross-day ROC-AUC matrix")
display(cross_auc.round(4))


In [ ]:

# ================================================================
# 10. DANN: PURGED TARGET ADAPTATION / HELD-OUT TEST
# ================================================================
def target_adaptation_positions(n):
    all_pos = valid_window_positions(n, np.arange(n))

    cut = int(len(all_pos) * DANN_ADAPT_RATIO)

    adapt = all_pos[:cut]

    # Purge sequence-overlap context from the test boundary.
    test_start = min(cut + PURGE_WINDOWS, len(all_pos))
    test = all_pos[test_start:]

    return (
        cap_positions(adapt, MAX_ADAPT_WINDOWS),
        cap_positions(test, MAX_TEST_WINDOWS)
    )

def evaluate_model_on_positions(model, d, positions, dann=False, threshold=0.5):
    loader = make_loader(d, positions)
    y, p = predict(model, loader, dann=dann)
    return y, p, metric_dict(y, p, threshold)

def run_dann_pair(source_day, target_day, lambda_domain=LAMBDA_DOMAIN,
                  save_prefix=None):

    src = ARTIFACTS[source_day]

    # Target uses source-only preprocessing.
    tgt = prepare_day(
        target_day,
        source_feature_cols=src["features"],
        source_scaler=src["scaler"],
        source_medians=src["medians"],
        fit_scaler=False
    )

    adapt_pos, test_pos = target_adaptation_positions(len(tgt["df"]))

    src_loader = make_loader(src, src["train_pos"], shuffle=True)
    tgt_adapt_loader = make_loader(tgt, adapt_pos, shuffle=True)

    # Baseline on EXACTLY the same held-out target partition.
    base = ForensicTransformer(len(src["features"])).to(DEVICE)
    base.load_state_dict(
        torch.load(
            OUT / f"baseline_{source_day}.pt",
            map_location=DEVICE
        )["model_state_dict"]
    )

    y_base, p_base, _ = evaluate_model_on_positions(
        base, tgt, test_pos,
        dann=False,
        threshold=SOURCE_THRESHOLDS[source_day]
    )

    base_metrics = metric_dict(
        y_base, p_base, SOURCE_THRESHOLDS[source_day]
    )

    # DANN starts from the trained source model.
    dann = DANNTransformer(len(src["features"])).to(DEVICE)
    dann.backbone.load_state_dict(base.state_dict())

    prefix = save_prefix or f"dann_{source_day}_to_{target_day}"
    ckpt = OUT / f"{prefix}.pt"

    history = train_dann(
        dann,
        src_loader,
        tgt_adapt_loader,
        src["y"][src["train_pos"]],
        str(ckpt),
        lambda_domain
    )

    # IMPORTANT:
    # Recalibrate threshold using SOURCE VALIDATION LABELS only.
    # Target test labels are still untouched.
    src_val_loader = make_loader(src, src["val_pos"])
    y_sv, p_sv = predict(dann, src_val_loader, dann=True)
    adapted_threshold, source_val_f1 = best_threshold(y_sv, p_sv)

    y_adapt, p_adapt, _ = evaluate_model_on_positions(
        dann, tgt, test_pos,
        dann=True,
        threshold=adapted_threshold
    )

    adapt_metrics = metric_dict(
        y_adapt, p_adapt, adapted_threshold
    )

    row = {
        "source_day": source_day,
        "target_day": target_day,
        "lambda_domain": lambda_domain,
        "baseline_f1": base_metrics["f1"],
        "baseline_roc_auc": base_metrics["roc_auc"],
        "baseline_pr_auc": base_metrics["pr_auc"],
        "adapted_f1": adapt_metrics["f1"],
        "adapted_roc_auc": adapt_metrics["roc_auc"],
        "adapted_pr_auc": adapt_metrics["pr_auc"],
        "adapted_precision": adapt_metrics["precision"],
        "adapted_recall": adapt_metrics["recall"],
        "f1_delta": adapt_metrics["f1"] - base_metrics["f1"],
        "roc_auc_delta": adapt_metrics["roc_auc"] - base_metrics["roc_auc"],
        "pr_auc_delta": adapt_metrics["pr_auc"] - base_metrics["pr_auc"],
        "source_val_threshold": adapted_threshold,
        "source_val_f1": source_val_f1,
        "adaptation_windows": len(adapt_pos),
        "test_windows": len(test_pos)
    }

    pd.DataFrame(history).to_csv(
        OUT / f"{prefix}_history.csv", index=False
    )

    return row, dann, tgt, test_pos, history

dann_rows = []
DANN_MODELS = {}
DANN_TEST_DATA = {}
DANN_HISTORIES = {}

if RUN_DANN:
    print("\n" + "="*90)
    print("PRIMARY DANN EXPERIMENT")
    print("="*90)

    for source_day, target_day in ADAPTATION_PAIRS:
        print(f"\nDANN {source_day} -> {target_day}")

        try:
            row, model, tgt, test_pos, hist = run_dann_pair(
                source_day, target_day,
                lambda_domain=LAMBDA_DOMAIN
            )

            dann_rows.append(row)
            DANN_MODELS[(source_day,target_day)] = model
            DANN_TEST_DATA[(source_day,target_day)] = (tgt, test_pos)
            DANN_HISTORIES[(source_day,target_day)] = hist

            print(
                f"Baseline F1={row['baseline_f1']:.4f} | "
                f"DANN F1={row['adapted_f1']:.4f} | "
                f"ΔF1={row['f1_delta']:+.4f}"
            )

        except Exception as e:
            print("DANN FAILED:", source_day, target_day, e)

dann_df = pd.DataFrame(dann_rows)
dann_df.to_csv(OUT / "domain_adaptation_results.csv", index=False)

print("\nPRIMARY DANN RESULTS")
display(dann_df)


In [ ]:

# ================================================================
# 11. DANN LAMBDA SENSITIVITY
# ================================================================
sweep_rows = []

# To keep runtime practical, use the first two representative pairs.
SWEEP_PAIRS = ADAPTATION_PAIRS[:2]

if RUN_DANN_SWEEP:
    print("\n" + "="*90)
    print("DANN LAMBDA SENSITIVITY")
    print("="*90)

    for source_day, target_day in SWEEP_PAIRS:
        for lam in DANN_LAMBDA_SWEEP:
            print(f"\nLambda={lam:.2f}: {source_day} -> {target_day}")

            try:
                row, _, _, _, _ = run_dann_pair(
                    source_day, target_day,
                    lambda_domain=lam,
                    save_prefix=f"sweep_lambda_{lam:.2f}_{source_day}_to_{target_day}"
                )
                sweep_rows.append(row)
            except Exception as e:
                print("SWEEP FAILED:", e)

sweep_df = pd.DataFrame(sweep_rows)
sweep_df.to_csv(OUT / "dann_lambda_sensitivity.csv", index=False)

if len(sweep_df):
    display(
        sweep_df[
            ["source_day","target_day","lambda_domain",
             "adapted_f1","adapted_roc_auc","adapted_pr_auc"]
        ]
    )


In [ ]:

# ================================================================
# 12. SUPERVISED TARGET FINE-TUNING — UPPER-BOUND COMPARATOR
# ================================================================
# This is NOT part of the unsupervised DANN claim.
# It answers: how much performance is recoverable if target labels
# are available? It therefore serves as an upper-bound / comparator.

def supervised_target_finetune(source_day, target_day):
    src = ARTIFACTS[source_day]

    tgt = prepare_day(
        target_day,
        source_feature_cols=src["features"],
        source_scaler=src["scaler"],
        source_medians=src["medians"],
        fit_scaler=False
    )

    adapt_pos, test_pos = target_adaptation_positions(len(tgt["df"]))

    model = ForensicTransformer(len(src["features"])).to(DEVICE)
    model.load_state_dict(
        torch.load(
            OUT / f"baseline_{source_day}.pt",
            map_location=DEVICE
        )["model_state_dict"]
    )

    loader = make_loader(tgt, adapt_pos, shuffle=True)
    test_loader = make_loader(tgt, test_pos)

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=FINE_TUNE_LR,
        weight_decay=1e-4
    )
    crit = nn.CrossEntropyLoss(
        weight=class_weights(tgt["y"][adapt_pos])
    )

    for epoch in range(1, FINE_TUNE_EPOCHS + 1):
        model.train()

        for xb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)
            loss = crit(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

        print(
            f"Target fine-tune {source_day}->{target_day} "
            f"epoch {epoch}/{FINE_TUNE_EPOCHS}"
        )

    # Threshold selected on the labelled target adaptation partition.
    adapt_eval_loader = make_loader(tgt, adapt_pos)
    ya, pa = predict(model, adapt_eval_loader)
    threshold, _ = best_threshold(ya, pa)

    yt, pt = predict(model, test_loader)
    m = metric_dict(yt, pt, threshold)

    return {
        "source_day": source_day,
        "target_day": target_day,
        "fine_tuned_f1": m["f1"],
        "fine_tuned_roc_auc": m["roc_auc"],
        "fine_tuned_pr_auc": m["pr_auc"],
        "threshold": threshold
    }

ft_rows = []

if RUN_SUPERVISED_TARGET_FT:
    print("\n" + "="*90)
    print("SUPERVISED TARGET FINE-TUNING COMPARATOR")
    print("="*90)

    for pair in ADAPTATION_PAIRS:
        try:
            ft_rows.append(
                supervised_target_finetune(*pair)
            )
        except Exception as e:
            print("Fine-tuning failed:", pair, e)

ft_df = pd.DataFrame(ft_rows)
ft_df.to_csv(OUT / "supervised_target_finetuning.csv", index=False)

if len(ft_df):
    display(ft_df)


In [ ]:

# ================================================================
# 13. VISUALISATION — BASELINE, CROSS-DAY, DANN
# ================================================================
def savefig(name):
    path = OUT / name
    plt.tight_layout()
    plt.savefig(path, dpi=180, bbox_inches="tight")
    plt.show()
    plt.close()
    return path

if RUN_PLOTS and len(baseline_df):
    # Baseline F1
    plt.figure(figsize=(9,5))
    plt.bar(baseline_df["day"], baseline_df["f1"])
    plt.xticks(rotation=45, ha="right")
    plt.ylabel("F1")
    plt.title("Same-Day Transformer Baseline F1")
    savefig("01_baseline_f1.png")

    # Baseline ROC-AUC / PR-AUC
    x = np.arange(len(baseline_df))
    width = 0.36
    plt.figure(figsize=(10,5))
    plt.bar(x-width/2, baseline_df["roc_auc"], width, label="ROC-AUC")
    plt.bar(x+width/2, baseline_df["pr_auc"], width, label="PR-AUC")
    plt.xticks(x, baseline_df["day"], rotation=45, ha="right")
    plt.ylim(0,1.05)
    plt.ylabel("Score")
    plt.title("Same-Day Baseline ROC-AUC and PR-AUC")
    plt.legend()
    savefig("02_baseline_auc_pr.png")

if RUN_PLOTS and len(cross_f1):
    plt.figure(figsize=(9,7))
    im = plt.imshow(cross_f1.values, vmin=0, vmax=1, aspect="auto")
    plt.colorbar(im, label="F1")
    plt.xticks(range(len(cross_f1.columns)), cross_f1.columns, rotation=45, ha="right")
    plt.yticks(range(len(cross_f1.index)), cross_f1.index)
    plt.xlabel("Test day")
    plt.ylabel("Training day")
    plt.title("Cross-Day F1 Matrix")
    for i in range(len(cross_f1.index)):
        for j in range(len(cross_f1.columns)):
            v = cross_f1.iloc[i,j]
            if pd.notna(v):
                plt.text(j, i, f"{v:.2f}", ha="center", va="center")
    savefig("03_cross_day_f1_heatmap.png")

    plt.figure(figsize=(9,7))
    im = plt.imshow(cross_auc.values, vmin=0, vmax=1, aspect="auto")
    plt.colorbar(im, label="ROC-AUC")
    plt.xticks(range(len(cross_auc.columns)), cross_auc.columns, rotation=45, ha="right")
    plt.yticks(range(len(cross_auc.index)), cross_auc.index)
    plt.xlabel("Test day")
    plt.ylabel("Training day")
    plt.title("Cross-Day ROC-AUC Matrix")
    for i in range(len(cross_auc.index)):
        for j in range(len(cross_auc.columns)):
            v = cross_auc.iloc[i,j]
            if pd.notna(v):
                plt.text(j, i, f"{v:.2f}", ha="center", va="center")
    savefig("04_cross_day_roc_auc_heatmap.png")

if RUN_PLOTS and len(dann_df):
    labels = [
        f"{r.source_day}->{r.target_day}"
        for r in dann_df.itertuples()
    ]

    x = np.arange(len(labels))
    width = 0.36

    plt.figure(figsize=(11,5))
    plt.bar(x-width/2, dann_df["baseline_f1"], width, label="Baseline")
    plt.bar(x+width/2, dann_df["adapted_f1"], width, label="DANN")
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylim(0,1)
    plt.ylabel("F1")
    plt.title("Baseline vs DANN F1 on Held-Out Target Test")
    plt.legend()
    savefig("05_baseline_vs_dann_f1.png")

    plt.figure(figsize=(11,5))
    plt.bar(x-width/2, dann_df["baseline_roc_auc"], width, label="Baseline")
    plt.bar(x+width/2, dann_df["adapted_roc_auc"], width, label="DANN")
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylim(0,1)
    plt.ylabel("ROC-AUC")
    plt.title("Baseline vs DANN ROC-AUC")
    plt.legend()
    savefig("06_baseline_vs_dann_roc_auc.png")

    plt.figure(figsize=(11,5))
    plt.bar(x, dann_df["f1_delta"])
    plt.axhline(0, linewidth=1)
    plt.xticks(x, labels, rotation=45, ha="right")
    plt.ylabel("DANN F1 − baseline F1")
    plt.title("DANN F1 Improvement / Degradation")
    savefig("07_dann_f1_delta.png")

if RUN_PLOTS and len(sweep_df):
    for pair in SWEEP_PAIRS:
        q = sweep_df[
            (sweep_df.source_day == pair[0]) &
            (sweep_df.target_day == pair[1])
        ]
        if len(q):
            plt.figure(figsize=(7,4))
            plt.plot(q["lambda_domain"], q["adapted_f1"], marker="o")
            plt.xlabel("Domain-loss strength λ")
            plt.ylabel("Target-test F1")
            plt.title(f"DANN λ sensitivity: {pair[0]} → {pair[1]}")
            savefig(
                f"08_lambda_{pair[0]}_to_{pair[1]}.png"
            )

print("Main visualisations generated.")


In [ ]:

# ================================================================
# 14. TRAINING CURVES
# ================================================================
if RUN_PLOTS:
    for day, hist in BASELINE_HISTORY.items():
        h = pd.DataFrame(hist)

        plt.figure(figsize=(7,4))
        plt.plot(h["epoch"], h["train_loss"], marker="o", label="Train")
        plt.plot(h["epoch"], h["val_loss"], marker="o", label="Validation")
        plt.xlabel("Epoch")
        plt.ylabel("Cross-entropy loss")
        plt.title(f"Transformer training: {day}")
        plt.legend()
        savefig(f"training_baseline_{day}.png")

    for pair, hist in DANN_HISTORIES.items():
        h = pd.DataFrame(hist)

        plt.figure(figsize=(7,4))
        plt.plot(h["epoch"], h["loss"], marker="o")
        plt.xlabel("Epoch")
        plt.ylabel("DANN loss")
        plt.title(f"DANN training: {pair[0]} → {pair[1]}")
        savefig(
            f"training_dann_{pair[0]}_to_{pair[1]}.png"
        )

print("Training curves generated.")


In [ ]:

# ================================================================
# 15. ROC / PR CURVES + CONFUSION MATRICES
# ================================================================
def plot_evaluation_curves(model, d, positions, name, dann=False, threshold=0.5):
    loader = make_loader(d, positions)
    y, p = predict(model, loader, dann=dann)

    # ROC
    if len(np.unique(y)) == 2:
        fpr, tpr, _ = roc_curve(y, p)
        auc = roc_auc_score(y, p)

        plt.figure(figsize=(6,5))
        plt.plot(fpr, tpr, label=f"AUC={auc:.3f}")
        plt.plot([0,1], [0,1], linestyle="--")
        plt.xlabel("False positive rate")
        plt.ylabel("True positive rate")
        plt.title(f"ROC curve — {name}")
        plt.legend()
        savefig(f"roc_{name}.png")

        # PR
        precision, recall, _ = precision_recall_curve(y, p)
        ap = average_precision_score(y, p)

        plt.figure(figsize=(6,5))
        plt.plot(recall, precision, label=f"AP={ap:.3f}")
        plt.xlabel("Recall")
        plt.ylabel("Precision")
        plt.title(f"Precision–Recall curve — {name}")
        plt.legend()
        savefig(f"pr_{name}.png")

    pred = (p >= threshold).astype(int)
    cm = confusion_matrix(y, pred, labels=[0,1])

    plt.figure(figsize=(5,4))
    plt.imshow(cm, aspect="auto")
    plt.colorbar(label="Count")
    plt.xticks([0,1], ["Benign","Anomaly"])
    plt.yticks([0,1], ["Benign","Anomaly"])
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion matrix — {name}")

    for i in range(2):
        for j in range(2):
            plt.text(j, i, str(cm[i,j]), ha="center", va="center")

    savefig(f"cm_{name}.png")

# One representative baseline and every primary DANN pair.
if RUN_PLOTS and RUN_BASELINE:
    d = ARTIFACTS[DAYS[0]]
    plot_evaluation_curves(
        BASELINE_MODELS[DAYS[0]],
        d,
        d["test_pos"],
        f"baseline_{DAYS[0]}",
        dann=False,
        threshold=SOURCE_THRESHOLDS[DAYS[0]]
    )

if RUN_PLOTS and RUN_DANN:
    for pair, model in DANN_MODELS.items():
        tgt, test_pos = DANN_TEST_DATA[pair]

        # Find threshold from the primary DANN result table.
        row = dann_df[
            (dann_df.source_day == pair[0]) &
            (dann_df.target_day == pair[1])
        ].iloc[0]

        plot_evaluation_curves(
            model, tgt, test_pos,
            f"dann_{pair[0]}_to_{pair[1]}",
            dann=True,
            threshold=float(row["source_val_threshold"])
        )

print("ROC, PR and confusion-matrix figures generated.")


In [ ]:

# ================================================================
# 16. XAI — FEATURE + TEMPORAL ATTRIBUTIONS
# ================================================================
def run_xai(model, d, positions, name, dann=False, n_samples=XAI_SAMPLES):
    positions = cap_positions(np.asarray(positions), n_samples)

    if len(positions) == 0:
        return None

    class Wrapper(nn.Module):
        def __init__(self, base, is_dann):
            super().__init__()
            self.base = base
            self.is_dann = is_dann

        def forward(self, x):
            z = self.base(x)
            logits = z[0] if self.is_dann else z
            return torch.softmax(logits, dim=1)[:, 1]

    wrapper = Wrapper(model, dann).to(DEVICE).eval()
    ig = IntegratedGradientsTorch(wrapper)

    feature_attrs = []
    temporal_attrs = []
    records = []

    X = d["X"]
    y = d["y"]

    for pos in positions:
        pos = int(pos)
        start = pos - SEQ_LEN + 1

        xb = (
            torch.from_numpy(
                X[start:pos+1]
            ).float().unsqueeze(0).to(DEVICE)
        )

        try:
            attr = ig.attribute(
                xb,
                baselines=torch.zeros_like(xb),
                n_steps=IG_STEPS
            )[0].cpu().numpy()
        except Exception as e:
            print("XAI failure:", e)
            continue

        # Average absolute attribution:
        # feature importance = average over time
        # temporal importance = average over features
        feature_attrs.append(np.mean(np.abs(attr), axis=0))
        temporal_attrs.append(np.mean(np.abs(attr), axis=1))

        with torch.no_grad():
            probability = float(wrapper(xb).cpu().item())

        records.append({
            "event_index": pos,
            "event_id": d["df"].iloc[pos]["event_id"],
            "true_label": int(y[pos]),
            "prediction_probability": probability
        })

    if not feature_attrs:
        return None

    feature_imp = np.mean(
        np.stack(feature_attrs), axis=0
    )
    temporal_imp = np.mean(
        np.stack(temporal_attrs), axis=0
    )

    fdf = pd.DataFrame({
        "feature": d["features"],
        "mean_abs_integrated_gradient": feature_imp
    }).sort_values(
        "mean_abs_integrated_gradient",
        ascending=False
    )

    tdf = pd.DataFrame({
        "sequence_position": np.arange(1, SEQ_LEN+1),
        "mean_abs_integrated_gradient": temporal_imp
    })

    rdf = pd.DataFrame(records)

    fdf.to_csv(
        OUT / f"xai_{name}_feature_importance.csv",
        index=False
    )
    tdf.to_csv(
        OUT / f"xai_{name}_temporal_importance.csv",
        index=False
    )
    rdf.to_csv(
        OUT / f"xai_{name}_samples.csv",
        index=False
    )

    # Feature plot
    top = fdf.head(15).sort_values(
        "mean_abs_integrated_gradient"
    )

    plt.figure(figsize=(9,6))
    plt.barh(
        top["feature"],
        top["mean_abs_integrated_gradient"]
    )
    plt.xlabel("Mean absolute Integrated Gradients")
    plt.title(f"Top XAI features — {name}")
    savefig(f"xai_{name}_features.png")

    # Temporal plot
    plt.figure(figsize=(9,4))
    plt.plot(
        tdf["sequence_position"],
        tdf["mean_abs_integrated_gradient"],
        marker="o"
    )
    plt.xlabel("Position in 32-event sequence")
    plt.ylabel("Mean absolute attribution")
    plt.title(f"Temporal XAI — {name}")
    savefig(f"xai_{name}_temporal.png")

    return fdf, tdf, rdf

XAI_RESULTS = {}

if RUN_XAI:
    # Representative same-day baseline
    day = DAYS[0]
    XAI_RESULTS[f"baseline_{day}"] = run_xai(
        BASELINE_MODELS[day],
        ARTIFACTS[day],
        ARTIFACTS[day]["test_pos"],
        f"baseline_{day}",
        dann=False
    )

    # All primary DANN models
    for pair, model in DANN_MODELS.items():
        tgt, test_pos = DANN_TEST_DATA[pair]
        key = f"dann_{pair[0]}_to_{pair[1]}"
        XAI_RESULTS[key] = run_xai(
            model,
            tgt,
            test_pos,
            key,
            dann=True
        )

print("XAI outputs generated.")


In [ ]:

# ================================================================
# 17. XAI SUMMARY TABLE
# ================================================================
xai_summary_rows = []

for name, result in XAI_RESULTS.items():
    if result is None:
        continue

    fdf, tdf, rdf = result

    top_features = fdf.head(10)["feature"].tolist()
    top_positions = (
        tdf.sort_values(
            "mean_abs_integrated_gradient",
            ascending=False
        )
        .head(5)["sequence_position"]
        .tolist()
    )

    xai_summary_rows.append({
        "model": name,
        "top_features": ", ".join(map(str, top_features)),
        "top_temporal_positions": ", ".join(map(str, top_positions))
    })

xai_summary = pd.DataFrame(xai_summary_rows)
xai_summary.to_csv(OUT / "xai_summary.csv", index=False)

display(xai_summary)


In [ ]:

# ================================================================
# 18. TEMPORAL FORENSIC GRAPH + TIMELINE VISUALISATION
# ================================================================
def build_temporal_graph(df, flagged_positions, window_seconds=WINDOW_SECONDS):
    flagged_positions = sorted(
        set(np.asarray(flagged_positions, dtype=int).tolist())
    )

    G = nx.DiGraph()

    ts_col = find_column(df.columns, TIMESTAMP_CANDIDATES)

    if ts_col:
        times = pd.to_datetime(
            df[ts_col], errors="coerce"
        ).astype("int64").to_numpy() / 1e9
    else:
        times = np.arange(len(df), dtype=float)

    for p in flagged_positions:
        G.add_node(
            int(p),
            event_id=str(df.iloc[p]["event_id"]),
            target=int(df.iloc[p]["target"]),
            timestamp=(
                str(df.iloc[p][ts_col])
                if ts_col else str(p)
            )
        )

    for a, b in zip(flagged_positions, flagged_positions[1:]):
        if np.isfinite(times[a]) and np.isfinite(times[b]):
            delta = times[b] - times[a]
            if 0 <= delta <= window_seconds:
                G.add_edge(
                    int(a), int(b),
                    relation="followed_by",
                    delta_seconds=float(delta)
                )

    return G

graph_rows = []

if RUN_GRAPHS and RUN_DANN:
    for pair, model in DANN_MODELS.items():

        tgt, test_pos = DANN_TEST_DATA[pair]

        row = dann_df[
            (dann_df.source_day == pair[0]) &
            (dann_df.target_day == pair[1])
        ].iloc[0]

        loader = make_loader(tgt, test_pos)
        y, p = predict(model, loader, dann=True)

        pred = (
            p >= float(row["source_val_threshold"])
        ).astype(int)

        flagged = test_pos[pred == 1]

        G = build_temporal_graph(
            tgt["df"],
            flagged,
            WINDOW_SECONDS
        )

        graph_name = f"graph_{pair[0]}_to_{pair[1]}"

        nx.write_graphml(
            G,
            OUT / f"{graph_name}.graphml"
        )

        graph_rows.append({
            "source_day": pair[0],
            "target_day": pair[1],
            "flagged_events": len(flagged),
            "graph_nodes": G.number_of_nodes(),
            "temporal_edges": G.number_of_edges()
        })

        # Timeline plot
        if len(flagged):
            ts_col = tgt["ts_col"]

            if ts_col:
                times = pd.to_datetime(
                    tgt["df"].iloc[flagged][ts_col],
                    errors="coerce"
                )
                times = times.dropna()

                plt.figure(figsize=(11,4))
                plt.scatter(
                    times,
                    np.ones(len(times))
                )
                plt.yticks([1], ["Flagged"])
                plt.xlabel("Timestamp")
                plt.title(
                    f"Forensic anomaly timeline — "
                    f"{pair[0]} → {pair[1]}"
                )
                savefig(
                    f"{graph_name}_timeline.png"
                )

graph_df = pd.DataFrame(graph_rows)
graph_df.to_csv(
    OUT / "forensic_graph_summary.csv",
    index=False
)

display(graph_df)


In [ ]:

# ================================================================
# 19. FINAL MASTER COMPARISON
# ================================================================
if len(dann_df):
    final_compare = dann_df[
        [
            "source_day","target_day",
            "baseline_f1","adapted_f1","f1_delta",
            "baseline_roc_auc","adapted_roc_auc","roc_auc_delta",
            "baseline_pr_auc","adapted_pr_auc","pr_auc_delta"
        ]
    ].copy()

    if len(ft_df):
        final_compare = final_compare.merge(
            ft_df[
                ["source_day","target_day",
                 "fine_tuned_f1","fine_tuned_roc_auc","fine_tuned_pr_auc"]
            ],
            on=["source_day","target_day"],
            how="left"
        )

    final_compare.to_csv(
        OUT / "final_model_comparison.csv",
        index=False
    )

    print("FINAL MODEL COMPARISON")
    display(final_compare.round(4))


In [ ]:

# ================================================================
# 20. FINAL VISUAL SUMMARY
# ================================================================
if len(dann_df):
    labels = [
        f"{r.source_day}->{r.target_day}"
        for r in dann_df.itertuples()
    ]

    x = np.arange(len(labels))
    width = 0.25

    plt.figure(figsize=(12,5))
    plt.bar(
        x - width,
        dann_df["baseline_f1"],
        width,
        label="Source-only baseline"
    )
    plt.bar(
        x,
        dann_df["adapted_f1"],
        width,
        label="DANN"
    )

    if "fine_tuned_f1" in final_compare.columns:
        plt.bar(
            x + width,
            final_compare["fine_tuned_f1"],
            width,
            label="Supervised target FT"
        )

    plt.xticks(
        x, labels,
        rotation=45,
        ha="right"
    )
    plt.ylim(0,1)
    plt.ylabel("F1")
    plt.title("Cross-Domain Model Comparison")
    plt.legend()
    savefig("09_final_model_comparison.png")

print("Final visual summary generated.")


In [ ]:

# ================================================================
# 21. MANUSCRIPT-READY SUMMARY / REPRODUCIBILITY MANIFEST
# ================================================================
summary = {
    "seed": SEED,
    "dataset_id": DATASET_ID,
    "days": DAYS,
    "sequence_length": SEQ_LEN,
    "sequence_stride": SEQ_STRIDE,
    "train_ratio": TRAIN_RATIO,
    "validation_ratio": VAL_RATIO,
    "test_ratio": TEST_RATIO,
    "model": {
        "type": "Transformer encoder",
        "d_model": D_MODEL,
        "heads": N_HEADS,
        "layers": N_LAYERS,
        "feed_forward": FF_DIM,
        "dropout": DROPOUT
    },
    "domain_adaptation": {
        "method": "DANN with gradient reversal",
        "primary_lambda": LAMBDA_DOMAIN,
        "lambda_sweep": DANN_LAMBDA_SWEEP,
        "target_adaptation_ratio": DANN_ADAPT_RATIO,
        "purge_windows": PURGE_WINDOWS,
        "target_test_labels_used_for_training": False
    },
    "xai": {
        "method": "Integrated Gradients implemented directly in PyTorch",
        "samples_per_explanation": XAI_SAMPLES,
        "integration_steps": IG_STEPS
    },
    "forensic_correlation": {
        "method": "deterministic temporal NetworkX graph",
        "window_seconds": WINDOW_SECONDS
    },
    "outputs": str(OUT)
}

with open(OUT / "experiment_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

# Create an easy-to-navigate manifest.
files = sorted(
    str(p.relative_to(OUT))
    for p in OUT.rglob("*")
    if p.is_file()
)

pd.DataFrame({"output_file": files}).to_csv(
    OUT / "OUTPUT_MANIFEST.csv",
    index=False
)

print("\n" + "="*90)
print("EXPERIMENT COMPLETE")
print("="*90)

print("Output directory:", OUT)
print("Number of generated files:", len(files))

print("\nKey outputs:")
for f in files:
    if any(x in f for x in [
        "results.csv", "matrix.csv", "comparison.csv",
        "summary.csv", ".png", "experiment_summary.json"
    ]):
        print(" ", f)

print("\nThe notebook has completed the baseline, cross-day, DANN, sensitivity,")
print("XAI, forensic graph and visualisation stages.")
